In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import Inception_V3_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,)

In [ ]:
#reduced random values to ensure reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
def prepare_cub_dataset():

    base = Path("/content/drive/MyDrive/Deep_Learning/Coursework/Test2")
    images_path = base / "Test"


    images = []
    with open(base / "images.txt", "r") as f:
        for line in f:
            parts = line.strip().split(" ", 1)
            if len(parts) == 2:
                img_id, filepath = parts
                images.append((int(img_id), filepath.strip()))


    labels = {}
    with open(base / "image_class_labels.txt", "r") as f:
        for line in f:
            img_id, class_id = line.strip().split()
            labels[int(img_id)] = int(class_id)


    data = []
    for img_id, filepath in images:
        if img_id in labels:
            data.append((filepath, labels[img_id]))

    full_dataset = datasets.ImageFolder(images_path)

    path_to_idx = {
        str(Path(p).relative_to(images_path)): i
        for i, (p, _) in enumerate(full_dataset.samples)}

    all_indices = []

    for filepath, class_id in data:
        if filepath in path_to_idx:
            all_indices.append(path_to_idx[filepath])

    return Subset(full_dataset, all_indices)

In [ ]:
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

In [ ]:
def create_dataloader(dataset, batch_size=32):

    eval_tfms = transforms.Compose([
        transforms.Resize(320),
        transforms.CenterCrop(299),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])])

    dataset = TransformSubset(dataset, eval_tfms)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False)

    size = len(loader.dataset)
    class_names = dataset.subset.dataset.classes

    return loader, size, class_names

In [ ]:
def build_inception(num_classes):
    model = models.inception_v3(
        weights=Inception_V3_Weights.DEFAULT,
        aux_logits=True)

    #replacing classifier with pre-trained one
    model.fc = torch.nn.Linear(
        model.fc.in_features,
        num_classes)


    if model.AuxLogits is not None:
        model.AuxLogits.fc = torch.nn.Linear(
            model.AuxLogits.fc.in_features,
            num_classes)

    return model

In [ ]:
def evaluate(model, dataloader, class_names, device, plot_cm=True):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
      for inputs, targets in dataloader:
          inputs = inputs.to(device)

          outputs = model(inputs)


          if isinstance(outputs, tuple):
              outputs = outputs[0]

          predictions = outputs.argmax(dim=1).cpu()

          all_preds.append(predictions)
          all_labels.append(targets)

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    print("\n===== TEST RESULTS =====")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f} (macro)")
    print(f"Recall   : {recall:.4f} (macro)")
    print(f"F1-score : {f1:.4f} (macro)")

    print("\nDetailed Classification Report:\n")
    print(classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        digits=4))

    if plot_cm:
        cm = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, cmap="Blues", square=True, cbar=True)
        plt.title("Confusion Matrix")
        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")
        plt.tight_layout()
        plt.show()

    return accuracy


In [ ]:
#---------main execution function------------
print("\nPreparing dataset...")
dataset = prepare_cub_dataset()

loader, size, class_names = create_dataloader(
    dataset, batch_size=32)
print("\nLoading saved model...")
load_path = "/content/drive/MyDrive/Deep_Learning/Coursework/saved_models/inception_cub_best.pth"

model = build_inception(len(class_names)).to(device)
state_dict = torch.load(load_path, map_location=device)
model.load_state_dict(state_dict)

print("\nEvaluating model...")
metrics = evaluate(
    model=model,
    dataloader=loader,
    class_names=class_names,
    device=device,
    plot_cm=True,)